In [2]:
import pandas as pd
df_meddra_all_se = pd.read_csv("meddra_all_se.tsv", sep="\t", header=None,
                 names=["CID","STITCH_CID", "UMLS_ID", "UMS_Concept_ID", "Source", "ADR_NAME"])
df_meddra_all_indications = pd.read_csv("meddra_all_indications.tsv", sep="\t", header=None,
                 names=["CID", "UMLS_ID", "text", "name", "name2","ID", "name4"])

In [13]:
df_meddra_all_se["new_CID"] = df_meddra_all_se["CID"].str.extract(r"CID(\d+)").astype(float).astype(int)
df_meddra_all_indications["new_CID"] = df_meddra_all_indications["CID"].str.extract(r"CID(\d+)").astype(float).astype(int)


In [21]:
drug_ID = pd.concat([df_meddra_all_se["new_CID"], df_meddra_all_indications["new_CID"]], axis=0)
drug_ID = drug_ID.unique()
len(drug_ID)
drug_ID

array([100000085, 100000119, 100000137, ..., 125102846, 144152182,
       146878492])

In [31]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

# 1. خوندن فایل meddra_all_indications.tsv
df_ind = pd.read_csv("meddra_all_indications.tsv", sep="\t", header=None)

# استخراج CID
df_ind["CID"] = df_ind[0].str.extract(r"CID(?:m)?(\d+)").astype(int)

# حذف تکراری‌ها
cid_list = df_ind["CID"].drop_duplicates().tolist()
print(f"📌 تعداد داروهای یکتا: {len(cid_list)}")

# 2. خوندن CID-SMILES.gz
df_smiles = pd.read_csv("CID-SMILES.gz", sep="\t", compression="gzip", names=["CID", "SMILES"])
df_smiles["CID"] = df_smiles["CID"].astype(int)
cid2smiles = dict(zip(df_smiles["CID"], df_smiles["SMILES"]))

# 3. ساخت fingerprint 881 بیتی
fingerprints = {}
for cid in cid_list:
    smiles = cid2smiles.get(cid)
    if smiles is None:
        print(f"⚠️ CID {cid} not found in CID-SMILES.gz")
        continue

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"⚠️ CID {cid} invalid SMILES")
        continue

    # Morgan fingerprint با طول 881
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=881)
    arr = np.array(fp, dtype=int).tolist()  # تبدیل به لیست
    fingerprints[cid] = arr

# 4. ساخت DataFrame با یک ستون fingerprint
df_fp = pd.DataFrame({
    "CID": list(fingerprints.keys()),
    "Fingerprint": list(fingerprints.values())
})

# 5. ذخیره CSV
df_fp.to_csv("indications_drug_fingerprints_881.csv", index=False)

print("✅ Done! Saved as indications_drug_fingerprints_881.csv")
print(df_fp.head())


📌 تعداد داروهای یکتا: 1437


[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerat

⚠️ CID 100001046 not found in CID-SMILES.gz
⚠️ CID 100002370 not found in CID-SMILES.gz
⚠️ CID 100002375 not found in CID-SMILES.gz
⚠️ CID 100002617 not found in CID-SMILES.gz
⚠️ CID 100002767 not found in CID-SMILES.gz
⚠️ CID 100002786 not found in CID-SMILES.gz
⚠️ CID 100002794 not found in CID-SMILES.gz
⚠️ CID 100003219 not found in CID-SMILES.gz
⚠️ CID 100003251 not found in CID-SMILES.gz
⚠️ CID 100003256 not found in CID-SMILES.gz
⚠️ CID 100003305 not found in CID-SMILES.gz
⚠️ CID 100003324 not found in CID-SMILES.gz
⚠️ CID 100003696 not found in CID-SMILES.gz
⚠️ CID 100004011 not found in CID-SMILES.gz
⚠️ CID 100004158 not found in CID-SMILES.gz


[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerat

⚠️ CID 100004456 not found in CID-SMILES.gz
⚠️ CID 100004614 not found in CID-SMILES.gz
⚠️ CID 100004675 not found in CID-SMILES.gz
⚠️ CID 100004812 not found in CID-SMILES.gz
⚠️ CID 100004828 not found in CID-SMILES.gz
⚠️ CID 100004893 not found in CID-SMILES.gz
⚠️ CID 100004992 not found in CID-SMILES.gz
⚠️ CID 100005005 not found in CID-SMILES.gz
⚠️ CID 100005078 not found in CID-SMILES.gz
⚠️ CID 100005468 not found in CID-SMILES.gz
⚠️ CID 100005479 not found in CID-SMILES.gz
⚠️ CID 100005515 not found in CID-SMILES.gz
⚠️ CID 100005538 not found in CID-SMILES.gz
⚠️ CID 100012453 not found in CID-SMILES.gz
⚠️ CID 100012559 not found in CID-SMILES.gz
⚠️ CID 100019090 not found in CID-SMILES.gz
⚠️ CID 100040976 not found in CID-SMILES.gz
⚠️ CID 100047319 not found in CID-SMILES.gz
⚠️ CID 100065628 not found in CID-SMILES.gz
⚠️ CID 100068613 not found in CID-SMILES.gz
⚠️ CID 100068740 not found in CID-SMILES.gz
⚠️ CID 100068844 not found in CID-SMILES.gz
⚠️ CID 100069512 not found in CI

[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerator
[12:20:28] DEPRECATION WARNING: please use MorganGenerat

⚠️ CID 100151165 not found in CID-SMILES.gz
⚠️ CID 106333887 not found in CID-SMILES.gz
⚠️ CID 122834577 not found in CID-SMILES.gz
⚠️ CID 124762228 not found in CID-SMILES.gz
⚠️ CID 142611257 not found in CID-SMILES.gz
⚠️ CID 144201343 not found in CID-SMILES.gz
⚠️ CID 144564107 not found in CID-SMILES.gz
⚠️ CID 144567678 not found in CID-SMILES.gz
⚠️ CID 146878492 not found in CID-SMILES.gz
⚠️ CID 151508717 not found in CID-SMILES.gz
⚠️ CID 151601240 not found in CID-SMILES.gz


[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerator
[12:20:29] DEPRECATION WARNING: please use MorganGenerat

✅ Done! Saved as indications_drug_fingerprints_881.csv
         CID                                        Fingerprint
0  100000085  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
1  100000119  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
2  100000137  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
3  100000143  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
4  100000158  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...


In [ ]:
import pandas as pd
import numpy as np
import ast  # برای تبدیل رشته لیست به لیست پایتونی

# خواندن CSV
df_fp = pd.read_csv("indications_drug_fingerprints_881.csv")
drug_list = df_fp["CID"].unique()

# انتخاب یک CID مشخص
cid = 100000137
row = df_fp[df_fp["CID"] == cid]["Fingerprint"].values[0]

# تبدیل رشته به لیست پایتونی
fingerprint_list = ast.literal_eval(row)

# تبدیل به numpy array
fingerprint_array = np.array(fingerprint_list, dtype=int)

print(f"Fingerprint shape: {fingerprint_array.shape}")
print(type(fingerprint_array))


[100000085 100000119 100000137 ... 170683024 170695640 171306834]
Fingerprint shape: (881,)
<class 'numpy.ndarray'>
